# Hybrid Retrieval and Fusion: BM25 + Dense Embeddings with RRF, and Learned Sparse (SPLADE)

## Overview
This notebook combines lexical keyword search with dense semantic retrieval on the **course RAG corpus (HotpotQA, built in `6.5`)**. It builds on the Week 3 encoders (S-BERT, BGE). Notebook `6.13` then uses hybrid search and re-ranking inside a full RAG pipeline.
1. **Lexical vs. Semantic Retrieval**: Comparing failure modes of BM25 and dense bi-encoders.
2. **Dual Retriever Setup**: BM25 and dense (Qdrant) retrievers over the same passages.
3. **Reciprocal Rank Fusion (RRF)**: Implementing rank-based fusion from scratch.
4. **Failure Modes**: The questions where BM25 beats dense, and the ones where dense beats BM25.
5. **Benchmark Comparison**: BM25, dense and hybrid hit@3, and what the numbers say.
6. **Modern Learned Sparse Retrievers**: SPLADE and BGE-M3.

> A hand-labelled research-paper version of this comparison (exact-term queries like *"WMT 2014 BLEU 28.4"*) is the take-home `Exercise-6.1.md`.

In [ ]:
%pip install -q rank_bm25 langchain-qdrant langchain-huggingface qdrant-client sentence-transformers

In [ ]:
import re

import numpy as np
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi

import rag_common as rc

cfg = rc.load_config()
rc.check_services(cfg, llm=False)
qa_pairs, passages = rc.load_hotpotqa(cfg)
client, embeddings, doc_store = rc.connect_qdrant(cfg)
print(f"{len(passages)} passages | {len(qa_pairs)} questions")

---
## 1. Lexical vs. Semantic Search

Every production retrieval engine faces two opposing query regimes:

| Query type | Best retriever | Why? |
|---|---|---|
| **Exact term**: names, IDs, numbers, acronyms | **BM25** (lexical) | Matches rare tokens exactly. Dense vectors smooth out specific names and digits. |
| **Paraphrase**: the question and the passage share few words | **Dense bi-encoder** | Matches the underlying meaning even with little word overlap. |
| **Real questions**: both an entity name and a concept | **Hybrid** | Needs the exact entity *and* the semantic intent. |

HotpotQA questions are full of entity names, so we can watch both regimes on real data.

---
## 2. Dual Retrievers

Two rankers over the same passages, both returning passage **titles** (unique per passage):
1. **BM25** (`rank_bm25.BM25Okapi`): lexical, built in memory.
2. **Dense**: the Qdrant collection from `6.5`.

In [ ]:
titles = [p["title"] for p in passages]
texts = [p["text"] for p in passages]
text_of = dict(zip(titles, texts))

def tokenize_text(text):
    return re.findall(r"\w+", text.lower())

bm25_index = BM25Okapi([tokenize_text(t) for t in texts])

def rank_bm25(query, k=3):
    return [titles[i] for i in np.argsort(-bm25_index.get_scores(tokenize_text(query)))[:k]]

def rank_dense(query, k=3):
    return [d.metadata["title"] for d in doc_store.similarity_search(query, k=k)]

q = qa_pairs[0]["question"]
print("Q:", q, "| gold passages:", qa_pairs[0]["supporting_titles"])
print("BM25 top-3 :", rank_bm25(q))
print("Dense top-3:", rank_dense(q))

---
## 3. Reciprocal Rank Fusion (RRF)

When combining BM25 and dense retrieval, raw scores cannot be directly added because **BM25 scores are unbounded positive numbers ($0$ to $50+$)**, whereas **dense cosine similarities fall between $-1$ and $+1$**.

### Cormack et al. (2009): Reciprocal Rank Fusion
RRF is a score-free, rank-based combination technique that uses only the **relative ranks** of documents from each retriever:
$$\text{RRF\_Score}(d) = \sum_{m \in M} \frac{1}{k + \text{rank}_m(d)}$$
where:
- $M = \{\text{BM25}, \text{Dense}\}$ is the set of retrievers.
- $\text{rank}_m(d) \in \{1, 2, 3, \dots\}$ is the 1-indexed rank position of document $d$ in system $m$.
- $k$ is a constant smoothing hyperparameter (standard convention: $k = 60$).

**Why $k=60$?** It ensures that top-5 rankings in either retriever contribute significantly to the fused score, while preventing lower-ranked noise from overpowering clear winners.

In [ ]:
def reciprocal_rank_fusion(bm25_ranks, dense_ranks, k=60, top_n=5):
    """Combine two ranked lists using Reciprocal Rank Fusion."""
    rrf_scores = {}
    for ranking in (bm25_ranks, dense_ranks):
        for rank_pos, doc_id in enumerate(ranking, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank_pos)
    sorted_docs = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [d for d, _ in sorted_docs[:top_n]], [s for _, s in sorted_docs[:top_n]]

def rank_hybrid(query, k=3, pool=20):
    fused, _ = reciprocal_rank_fusion(rank_bm25(query, pool), rank_dense(query, pool), top_n=k)
    return fused

print("Hybrid top-3:", rank_hybrid(q))

---
## 3. Failure Modes: Who Wins Where?

We score **hit@3** on all questions: does the answer text appear in the top-3 retrieved passages? Then we list the questions that **one retriever answers and the other misses**.

In [ ]:
def hit(rank_fn, qa, k=3):
    docs = [Document(page_content=text_of[t]) for t in rank_fn(qa["question"], k)]
    return rc.retrieval_hit(docs, qa["answer"])

retrievers = {"BM25": rank_bm25, "Dense": rank_dense, "Hybrid RRF": rank_hybrid}
hits = {name: [hit(fn, qa) for qa in qa_pairs] for name, fn in retrievers.items()}

only_bm25 = [qa for qa, b, d in zip(qa_pairs, hits["BM25"], hits["Dense"]) if b and not d]
only_dense = [qa for qa, b, d in zip(qa_pairs, hits["BM25"], hits["Dense"]) if d and not b]
print(f"{len(qa_pairs)} questions | BM25 found the answer and dense missed: {len(only_bm25)} | dense found it and BM25 missed: {len(only_dense)}\n")
print("BM25 wins:")
for qa in only_bm25[:3]:
    print(f"  Q: {qa['question']}\n     A: {qa['answer']}")
print("\nDense wins:")
for qa in only_dense[:3]:
    print(f"  Q: {qa['question']}\n     A: {qa['answer']}")

---
## 4. Benchmark Comparison

Each retriever rescues questions the other misses. **RRF** combines their rankings without needing to compare scores:

In [ ]:
print(f"{'Retriever':<14}{'Hit@3 (all questions)':>24}")
print("-" * 38)
for name, h in hits.items():
    print(f"{name:<14}{np.mean(h) * 100:>23.1f}%")

# The shared ledger scores every technique on the same fixed sample (rag_common.eval_set), hit only (no LLM here)
sample = rc.eval_set(qa_pairs)
for label, fn in [("BM25 top-3", rank_bm25), ("Dense top-3 (baseline)", rank_dense), ("Hybrid RRF top-3", rank_hybrid)]:
    rc.record_result(label, "retrieval", hit=float(np.mean([hit(fn, qa) for qa in sample])), n=len(sample))
print(f"\nRecorded BM25, dense and hybrid on the shared {len(sample)}-question sample (see rc.show_ablation() in 6.11).")

**Read it honestly.** Hybrid is not automatically the best retriever. On this corpus and embedding model, dense search alone beats the fused top-3, because BM25 adds noise to a short list. Fusion earns its keep as **candidate generation**: take a wider pool (top 10 to 20) and let a reranker choose, as in `6.9` and the capstone `6.13`. Judge a technique on the whole pipeline, not on one stage.

---
## 5. Modern Sparse Retrievers (SPLADE & BGE-M3)

### 5.1 SPLADE (Sparse Lexical and Expansion Model)
Rather than computing static TF-IDF, **SPLADE** passes input text through a BERT model and applies its Masked Language Modeling (MLM) head over the entire 30,000-word vocabulary:
- For each vocabulary token $w$, SPLADE predicts an importance weight.
- Synonyms and related terms not present in the document receive positive weights (**Neural Term Expansion**).
- Result: A sparse vector stored directly in a standard inverted index (Elasticsearch, Lucene, Vespa) that resolves the vocabulary mismatch problem while keeping inverted-index retrieval speeds!

### 5.2 BGE-M3 (BAAI Multi-Lingual, Multi-Function, Multi-Granularity)
BGE-M3 is the ultimate multi-task retrieval model. From a single model pass, it outputs **three distinct representations**:
1. **Dense Vector**: 1,024d embedding for broad semantic similarity.
2. **Sparse Vector**: Lexical weights over vocabulary for exact term matches (SPLADE-style).
3. **Multi-Vector**: Token embeddings for ColBERT-style MaxSim late interaction.

The unified score combines all three:
$$S_{\text{BGE-M3}}(q, d) = w_1 S_{\text{dense}}(q, d) + w_2 S_{\text{sparse}}(q, d) + w_3 S_{\text{colbert}}(q, d)$$

---
## 6. Summary & Exercises

### Question 1
Why does Reciprocal Rank Fusion (RRF) use rank positions instead of normalizing and averaging the raw scores from BM25 and dense bi-encoders?
<details>
<summary><b>View Answer</b></summary>
BM25 scores and dense cosine similarities have completely different distributions. BM25 is unbounded ($0$ to $\infty$) and sensitive to document length and collection frequency, whereas cosine similarity is bounded in $[-1, 1]$. Normalizing both via min-max or z-score depends heavily on outlier documents in each batch. RRF relies exclusively on relative rank order, making it scale-invariant, robust to score anomalies, and hyperparameter-free aside from the constant $k$.
</details>

### Question 2
What is the primary difference between BM25 and SPLADE for sparse lexical retrieval?
<details>
<summary><b>View Answer</b></summary>
BM25 relies on exact surface-form word matching using static inverse document frequency (IDF). SPLADE uses a Transformer MLM head to predict weights over the entire vocabulary, adding weights for relevant words that do not physically appear in the document (neural term expansion). Both produce sparse vectors queryable via an inverted index, but SPLADE overcomes vocabulary mismatch.
</details>

Next: `6.8-Retrieval-Query-Transformation.ipynb` rewrites the question before retrieval.
